In [ ]:
!pip install --upgrade langchain langchain-community langgraph openai langchain_openai wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.3/956.3 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.5/449.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 14.7 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha2

In [ ]:
!pip install -qU "langchain[google-genai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.7.0 which is incompatible.


In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [ ]:
wiki_tool.run({"query": "AI agents"})

'Page: Agentic AI\nSummary: Agentic AI is a class of artificial intelligence that focuses on autonomous systems that can make decisions and perform tasks with or without human intervention. The independent systems automatically respond to conditions, with procedural, algorithmic, and human-like creati'

In [ ]:
from langchain.agents import initialize_agent, Tool
from langchain.chat_models import init_chat_model

import os
import getpass

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

Enter API key for Google Gemini: ··········


In [ ]:
llm = init_chat_model("gemini-2.0-flash", model_provider="google_genai")


In [ ]:
tools = [wiki_tool]

# Tool binding
llm_with_tools = llm.bind_tools(tools)

#Tool calling
result = llm_with_tools.invoke("Hello world!")
result
result.content


'Hello! How can I help you today?'

In [ ]:
tools = [wiki_tool]

# Tool binding
llm_with_tools = llm.bind_tools(tools)

#Tool calling
result = llm_with_tools.invoke("hi how are you")
result
result.content


'I am doing well, thank you for asking. How can I help you today?'

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(llm, tools)


In [ ]:
from langchain_core.messages import HumanMessage

#First up, let's see how it responds when there's no need to call a tool:
response = agent_executor.invoke({"messages": [HumanMessage(content="hi!")]})

response["messages"]

[HumanMessage(content='hi!', additional_kwargs={}, response_metadata={}, id='cd789482-7551-4fdd-8166-8ac20a7e881c'),
 AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--2d7ab845-a773-4a81-8b58-23af8267a14d-0', usage_metadata={'input_tokens': 50, 'output_tokens': 11, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}})]

In [ ]:
print(response["messages"][-1].content)


Hi there! How can I help you today?


In [ ]:
from langchain_core.messages import HumanMessage

#First up, let's see how it responds when there's no need to call a tool:
response = agent_executor.invoke({"messages": [HumanMessage(content="what is agentic ai")]})

response["messages"]

[HumanMessage(content='what is agentic ai', additional_kwargs={}, response_metadata={}, id='b6b00396-69d5-4c17-9293-bed9aea26cbb'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'wikipedia', 'arguments': '{"query": "agentic AI"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--9d4fc836-42a7-4f10-bb1c-0c3109020873-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'agentic AI'}, 'id': 'bf7cc40e-01e1-4575-8a3e-6aba59043149', 'type': 'tool_call'}], usage_metadata={'input_tokens': 53, 'output_tokens': 5, 'total_tokens': 58, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='Page: Agentic AI\nSummary: Agentic AI is a class of artificial intelligence that focuses on autonomous systems that can make decisions and perform tasks with or without human intervention. The independent systems automatically respond to conditions, with

In [ ]:
print(response["messages"][-1].content)

Agentic AI is a class of artificial intelligence that focuses on autonomous systems that can make decisions and perform tasks with or without human intervention. The independent systems automatically respond to conditions, with procedural, algorithmic, and human-like creation.


In [ ]:
from langchain_core.messages import HumanMessage

#First up, let's see how it responds when there's no need to call a tool:
response = agent_executor.invoke({"messages": [HumanMessage(content="What is a Weather in Lahore")]})

print(response["messages"][-1].content)

Lahore has a five-season semi-arid climate, bordering a humid subtropical climate. The seasons are: foggy winter (1 December – 15 February) with few western disturbances causing rain; pleasant spring (16 February – 15 April).
